# Day 17 · Is retrieval any good? recall@k

**First-principles question:** How often does the right document even reach the model?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import json
import numpy as np
from lab.search import Index, article_chunks

index = Index(article_chunks())
QUERIES_FILE = ROOT / "data" / "acme" / "queries.jsonl"
queries = [json.loads(line) for line in QUERIES_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
print(len(queries), "queries. Add yours to", QUERIES_FILE)

## 1 · Where does the right article land?

In [ ]:
def rank_of_right_article(q, k=10):
    """1 if the right article is first, 2 if second, ... None if not in the top k."""
    for position, hit in enumerate(index.search(q["query"], k=k), start=1):
        if hit["article"] == q["article"]:
            return position
    return None

ranks = [rank_of_right_article(q) for q in queries]
print(list(zip([q["id"] for q in queries], ranks)))

## 2 · recall@k and MRR

In [ ]:
def recall_at_k(ranks, k):
    """Share of queries whose right article is in the top k."""
    # TODO: count ranks that are not None and <= k, divide by len(ranks)
    ...

def mrr(ranks):
    """Mean reciprocal rank: the average of 1 / rank (0 when the article was not found)."""
    # TODO
    ...

def se(p, n):
    return float(np.sqrt(p * (1 - p) / n))

for k in (1, 3, 5):
    r = recall_at_k(ranks, k)
    print(f"recall@{k}: {r:.0%} ± {2 * se(r, len(ranks)):.0%}")
print(f"MRR: {mrr(ranks):.2f}")

## 3 · The misses
Read each one. Why did search miss? Write one line each below.

In [ ]:
for q, r in zip(queries, ranks):
    if r is None or r > 3:
        top = [h["article"] for h in index.search(q["query"], k=3)]
        print(f"{q['id']} {q['query']!r}: right {q['article']}, rank {r}, top 3 {top}")

**Why each miss happened:**

- …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert recall_at_k([1, 2, None, 4], 2) == 0.5 and recall_at_k([1, 1], 1) == 1
assert abs(mrr([1, 2, None, 4]) - (1 + 0.5 + 0 + 0.25) / 4) < 1e-9
assert recall_at_k(ranks, 1) <= recall_at_k(ranks, 3) <= recall_at_k(ranks, 5)
assert len(queries) >= 25, f"Add query pairs to queries.jsonl: {len(queries)} of 25"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")